# 11주차 한국어 문장 감성 분류 — 사전학습 언어모델 파인튜닝 (3교시 실습) 〔빈칸본〕

**목표**: 사전학습된 한국어 언어모델 **KoELECTRA-small** 을 가져와 영화 리뷰(NSMC)의 **긍정/부정**에 맞게 파인튜닝하고,
저장한 모델로 **내 문장**을 판정한다. 그리고 10주차 LSTM 결과와 비교한다.

```
  1단계 데이터 불러오기 (NSMC 영화 리뷰)
  2단계 토크나이저 관찰 — 문장이 어떤 조각으로 나뉘나      ← 2교시 "서브워드 토크나이저"
  3단계 사전학습 모델 + 분류 헤드                          ← 2교시 "파인튜닝 = 전이학습의 텍스트판"
  4단계 학습 함수 + 최고 모델 저장                          ← 9주차와 같은 학습 루프
  5단계 평가 (혼동행렬 · 틀린 문장 보기)
  6단계 저장한 모델로 내 문장 판정 (pipeline)
  7단계 10주차 LSTM 과 비교
```

### 빈칸 채우는 법

- 코드 안의 **`____`** 를 지우고 알맞은 코드를 적습니다. 줄 전체가 `____` 이면 코드 한 줄입니다.
- 빈칸 바로 위의 **`# 빈칸 ⓝ` 힌트**를 먼저 읽으세요. 대부분 정답 형태가 힌트 안에 있습니다.
- 셀을 실행한 뒤 **"기대 출력"** 과 비교합니다. 비슷하면 통과입니다.
- 빈칸은 모두 **10곳 · 13줄**입니다.

> **기대 출력은 교수 PC(GPU 없음)에서 실행한 값입니다.** CPU 에서는 훈련 데이터를 줄여 학습하므로,
> 실습실 GPU 에서는 데이터가 더 많아 **정확도가 더 높게** 나옵니다. 숫자가 조금 달라도 흐름이 같으면 통과입니다.

---

## 0. 준비 — 라이브러리와 장치

처음 실행하면 사전학습 모델(약 54MB)을 내려받습니다. 실습실은 미리 받아 두었습니다.

In [ ]:
# 셀 0 — 준비  (빈칸 없음)
import os, time, random
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from transformers.utils import logging as hf_logging
import matplotlib.pyplot as plt

hf_logging.set_verbosity_error()                  # 긴 안내 메시지 줄이기
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

def set_seed(s=42):
    random.seed(s); torch.manual_seed(s)
set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 실습실 GPU 면 원래 크기로, CPU 면 줄여서 학습한다  ★ 숫자를 바꾸지 마세요
if device.type == "cuda":
    N_TRAIN, N_VAL, EPOCHS = 20000, 4000, 2
else:
    N_TRAIN, N_VAL, EPOCHS = 3000, 1000, 2

MODEL   = "monologg/koelectra-small-v3-discriminator"   # 한국어 사전학습 모델 (작은 판)
MAX_LEN = 64                                             # 문장을 최대 64 조각까지만
SAVE    = "models/nsmc-koelectra"                        # 학습한 모델을 저장할 폴더
print("장치 :", device, f"| 훈련 {N_TRAIN}건 / 검증 {N_VAL}건 / {EPOCHS} epoch")

---

## 1단계 — 데이터 불러오기 (NSMC)

**NSMC** 는 네이버 영화 리뷰 20만 건에 **긍정(1) / 부정(0)** 이 붙은 데이터입니다. 수업에서는 그중 일부만 씁니다.
10주차와 같은 파일(`data/nsmc_subset/`)을 그대로 씁니다.

In [ ]:
# 셀 1 — 데이터 불러오기  (빈칸 없음)
from preprocess_text import load_nsmc, make_subset

DATA = "data/nsmc_subset"
try:
    tr_x, tr_y, va_x, va_y = load_nsmc(DATA)
except FileNotFoundError:                          # 없으면 내려받아 만든다 (약 20MB)
    make_subset(DATA)
    tr_x, tr_y, va_x, va_y = load_nsmc(DATA)

tr_x, tr_y = tr_x[:N_TRAIN], tr_y[:N_TRAIN]
va_x, va_y = va_x[:N_VAL], va_y[:N_VAL]
LABELS = ["부정", "긍정"]

print(f"훈련 {len(tr_x)}건 (긍정 {sum(tr_y)}) / 검증 {len(va_x)}건 (긍정 {sum(va_y)})")
for t, y in list(zip(tr_x, tr_y))[:4]:
    print(f"  [{LABELS[y]}] {t}")

**기대 출력** (교수 PC · CPU)
```
훈련 3000건 (긍정 1537) / 검증 1000건 (긍정 506)
  [부정] 미국식유머??나도 미국식유머좋아하는데 이건진짜 재미없다..보다가 끔
  [부정] 언어를 통해 전염되는 좀비병? 설정이 너무 이상해
  [긍정] 좋은 영화인데 개봉을 너무 짧게해서 아쉬웠다.
  [긍정] 잘봤다
```

---

## 2단계 — 토크나이저 관찰

모델은 글자를 바로 읽지 못합니다. **토크나이저**가 문장을 **자주 나오는 조각(서브워드)** 으로 자르고 번호로 바꿉니다.
`##` 이 붙은 조각은 **앞 조각에 이어지는** 조각이라는 표시입니다.

> 10주차에는 띄어쓰기로만 잘랐기 때문에 처음 보는 단어는 전부 `<unk>` 이 되었습니다.
> 서브워드는 처음 보는 단어도 **아는 조각들로 나눠서** 표현합니다.

In [ ]:
# 셀 2 — 토크나이저 관찰
# 빈칸 ① (1줄) — 힌트 : 모델 이름으로 토크나이저를 불러온다.  AutoTokenizer.from_pretrained(MODEL)
tok = ____
print("어휘 크기 :", tok.vocab_size)

for s in ["이 영화 정말 재미없다", "핵노잼 ㅋㅋㅋ 시간낭비", "스토리는뭐랄까그냥그랬어요"]:
    # 빈칸 ② (1줄) — 힌트 : 문장 s 를 조각으로 자른다.  tok.tokenize(s)
    pieces = ____
    print(f"{s}\n   → {pieces}")

# 여러 문장을 한 번에 번호로 바꾸기 — 짧은 문장은 0(PAD)으로 채운다
batch = tok(["짧다", "이 영화는 정말 길고 지루하고 재미없었다"], padding=True, return_tensors="pt")
print("\ninput_ids      :\n", batch["input_ids"])
print("attention_mask :\n", batch["attention_mask"], " ← 1 = 진짜 조각, 0 = 채운 자리")
print("첫 문장 되돌리기 :", tok.convert_ids_to_tokens(batch["input_ids"][0]))

**기대 출력**
```
어휘 크기 : 35000
이 영화 정말 재미없다
   → ['이', '영화', '정말', '재미없', '##다']
핵노잼 ㅋㅋㅋ 시간낭비
   → ['핵', '##노', '##잼', 'ㅋㅋㅋ', '시간', '##낭', '##비']
스토리는뭐랄까그냥그랬어요
   → ['스토리', '##는', '##뭐', '##랄', '##까', '##그', '##냥', '##그', '##랬', '##어요']

input_ids      :
 tensor([[    2,  3365,  4176,     3,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0],
        [    2,  3240,  6394,  4034,  6595,  2139,  4219, 13070,  4279,  4219,
         22571,  4480,  4176,     3]])
attention_mask :
 tensor([[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])  ← 1 = 진짜 조각, 0 = 채운 자리
첫 문장 되돌리기 : ['[CLS]', '짧', '##다', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
```

> **확인**: 모든 문장의 앞에 **`[CLS]`**, 끝에 **`[SEP]`** 가 붙습니다. `[CLS]` 자리의 출력이 **문장 전체를 대표**해 분류에 쓰입니다(2교시).
> 짧은 문장은 `[PAD]`(번호 0)로 채우고, `attention_mask` 가 0 이면 모델이 그 자리를 **무시**합니다.

> **막히면**: `OSError ... monologg/...` — 모델을 못 받았습니다. 인터넷 연결 또는 실습실 공유 캐시를 확인하세요.

---

## 3단계 — 사전학습 모델 + 분류 헤드

먼저 모든 문장을 한 번에 번호로 바꿔 `DataLoader` 로 묶습니다(빈칸 없음).
그다음 **사전학습 모델 위에 긍정/부정 2개짜리 분류 헤드**를 붙입니다 — 9주차에 ResNet18 의 `fc` 를 바꾼 것과 같은 일입니다.

```
  [KoELECTRA]  문장 조각들 → [ 본체 : 문장의 뜻을 읽는 부분 (사전학습) ] → [CLS] 출력 → 새 분류 헤드 (2개)  ★ 새로 만든다
```

In [ ]:
# 셀 3 — 문장을 번호로 바꿔 DataLoader 만들기  (빈칸 없음)
def make_loader(texts, labels, shuffle):
    enc = tok(texts, padding="max_length", truncation=True, max_length=MAX_LEN, return_tensors="pt")
    ds = TensorDataset(enc["input_ids"], enc["attention_mask"], torch.tensor(labels))
    return DataLoader(ds, batch_size=32, shuffle=shuffle)

train_loader = make_loader(tr_x, tr_y, shuffle=True)
val_loader   = make_loader(va_x, va_y, shuffle=False)
ids, mask, yb = next(iter(train_loader))
print("배치 shape :", tuple(ids.shape), "  ← (문장 32개, 조각 64개)")

In [ ]:
# 셀 4 — 사전학습 모델 불러오기 + 분류 헤드
set_seed()
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    # 빈칸 ③ (1줄) — 힌트 : 분류할 종류의 수. 긍정/부정 두 가지
    num_labels=____,
    id2label={0: "부정", 1: "긍정"}, label2id={"부정": 0, "긍정": 1},   # 결과를 이름으로 보이게
).to(device)

print("분류 헤드 :", model.classifier)
print(f"전체 파라미터 : {sum(p.numel() for p in model.parameters()):,}")

with torch.no_grad():
    out = model(input_ids=ids[:2].to(device), attention_mask=mask[:2].to(device))
# 빈칸 ④ (1줄) — 힌트 : HuggingFace 모델의 출력은 객체다. 점수는 .logits 에 들어 있다
logits = ____
print("logits shape :", tuple(logits.shape), "  ← (문장 2개, 종류 2개)")

**기대 출력 (셀 3 · 셀 4)**
```
배치 shape : (32, 64)   ← (문장 32개, 조각 64개)
분류 헤드 : ElectraClassificationHead(
  (dense): Linear(in_features=256, out_features=256, bias=True)
  (activation): GELUActivation()
  (dropout): Dropout(p=0.1, inplace=False)
  (out_proj): Linear(in_features=256, out_features=2, bias=True)
)
전체 파라미터 : 14,122,498
logits shape : (2, 2)   ← (문장 2개, 종류 2개)
```

> **확인 ★**: 분류 헤드는 아직 **학습 전**이라 아무렇게나 맞힙니다. 본체(약 1,400만 개)는 이미 한국어를 읽을 줄 압니다.
> 9주차와 다른 점: 이번에는 본체를 **얼리지 않고 전부 조금씩** 고칩니다(작은 학습률 `5e-5`). 언어모델 파인튜닝의 표준 방식입니다.

> **함정 ★**: HuggingFace 모델은 `model(...)` 이 **텐서가 아니라 객체**를 돌려줍니다. 점수는 **`.logits`**, 정답을 함께 넣으면 손실은 **`.loss`** 입니다.

---

## 4단계 — 학습 함수 + 최고 모델 저장

학습 루프는 9주차와 같습니다. 달라진 점은 두 가지뿐입니다.

- 모델에 **`labels=`** 를 함께 넣으면 손실을 **모델이 직접 계산**해 `out.loss` 로 돌려줍니다.
- 저장은 `torch.save` 대신 **`save_pretrained`** — 모델과 **토크나이저를 함께** 폴더로 저장합니다.

In [ ]:
# 셀 5 — 평가 함수와 학습 함수
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total, correct, n, preds = 0.0, 0, 0, []
    for ids, mask, yb in loader:
        ids, mask, yb = ids.to(device), mask.to(device), yb.to(device)
        out = model(input_ids=ids, attention_mask=mask, labels=yb)
        total += out.loss.item() * len(yb)
        p = out.logits.argmax(1)
        correct += (p == yb).sum().item(); n += len(yb); preds += p.cpu().tolist()
    return total / n, correct / n, preds


def fit(model, epochs, lr=5e-5):
    set_seed()
    # 빈칸 ⑤ (1줄) — 힌트 : 언어모델 파인튜닝의 표준 옵티마이저.  torch.optim.AdamW(model.parameters(), lr=lr)
    optimizer = ____
    hist = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_acc, t0 = 0.0, time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        run, n = 0.0, 0
        for step, (ids, mask, yb) in enumerate(train_loader, 1):
            ids, mask, yb = ids.to(device), mask.to(device), yb.to(device)
            # 빈칸 ⑥ (1줄) — 힌트 : 정답까지 함께 넣으면 손실을 모델이 계산해 준다.  labels=yb
            out = model(input_ids=ids, attention_mask=mask, labels=____)
            loss = out.loss
            # 빈칸 ⑦ (3줄) — 힌트 : 기울기 지우기 → 역전파 → 갱신 (5주차부터 같은 세 줄)
            ____
            ____
            ____
            run += loss.item() * len(yb); n += len(yb)
            if step % 30 == 0:
                print(f"   epoch {epoch} | {step}/{len(train_loader)} 배치 | 훈련 손실 {run/n:.4f}")

        val_loss, val_acc, _ = evaluate(model, val_loader)
        hist["train_loss"].append(run / n); hist["val_loss"].append(val_loss); hist["val_acc"].append(val_acc)
        mark = ""
        if val_acc > best_acc:
            best_acc = val_acc
            # 빈칸 ⑧ (2줄) — 힌트 : 모델과 토크나이저를 SAVE 폴더에 함께 저장
            #                     model.save_pretrained(SAVE)  /  tok.save_pretrained(SAVE)
            ____
            ____
            mark = "  ← 최고, 저장"
        print(f"epoch {epoch} | 훈련 손실 {run/n:.4f} | 검증 손실 {val_loss:.4f} | 검증 정확도 {val_acc*100:.2f}%{mark}")

    hist["best_acc"], hist["time"] = best_acc, time.time() - t0
    print(f"\n최고 검증 정확도 {best_acc*100:.2f}%  |  학습 시간 {hist['time']:.1f}초  |  저장 : {SAVE}")
    return hist

import inspect
if "____" in inspect.getsource(fit):
    print("⚠️ 빈칸 ⑤ ⑥ ⑦ ⑧ 중 아직 채우지 않은 곳이 있습니다")
else:
    print("함수 준비 완료 : evaluate / fit")

> **이 셀은 함수를 만들기만 합니다.** `함수 준비 완료` 가 나오면 통과입니다.

In [ ]:
# 셀 6 — 파인튜닝 실행  (빈칸 없음)
before = evaluate(model, val_loader)[1]
print(f"학습 전 검증 정확도 : {before*100:.2f}%   ← 헤드가 아직 아무것도 모른다\n")
hist = fit(model, EPOCHS)

ep = range(1, EPOCHS + 1)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(ep, hist["train_loss"], "o-", label="훈련 손실"); ax[0].plot(ep, hist["val_loss"], "o-", label="검증 손실")
ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3); ax[0].set_title("손실")
ax[1].plot(ep, [a * 100 for a in hist["val_acc"]], "o-", color="tab:green")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("%"); ax[1].grid(alpha=.3); ax[1].set_title("검증 정확도")
plt.tight_layout(); plt.show()

**기대 출력** (교수 PC CPU 실측 · 훈련 3,000건 · 학습 1008.1초 (다른 작업과 동시 실행))
```
학습 전 검증 정확도 : 47.20%   ← 헤드가 아직 아무것도 모른다

   epoch 1 | 30/94 배치 | 훈련 손실 0.6929
   epoch 1 | 60/94 배치 | 훈련 손실 0.6838
   epoch 1 | 90/94 배치 | 훈련 손실 0.6387
epoch 1 | 훈련 손실 0.6350 | 검증 손실 0.5045 | 검증 정확도 77.40%  ← 최고, 저장
   epoch 2 | 30/94 배치 | 훈련 손실 0.4683
   epoch 2 | 60/94 배치 | 훈련 손실 0.4634
   epoch 2 | 90/94 배치 | 훈련 손실 0.4527
epoch 2 | 훈련 손실 0.4510 | 검증 손실 0.4392 | 검증 정확도 80.10%  ← 최고, 저장

최고 검증 정확도 80.10%  |  학습 시간 1008.1초  |  저장 : models/nsmc-koelectra
```

> **관찰 ★**: 학습 전 **47.20%**(두 종류를 찍는 수준) → 1 epoch **77.40%** → 2 epoch **80.10%**. 훈련 손실과 검증 손실이 함께 내려가 과적합 신호는 없습니다. 첫 epoch 초반(30배치) 손실이 0.69 근처에 머무는 것은 새 분류 헤드가 자리를 잡는 중이기 때문입니다 — 조금 기다리면 내려갑니다.

> **막히면**
> | 증상 | 원인 · 조치 |
> |---|---|
> | `NameError: name '____'` | 빈칸을 채우지 않았다 |
> | `'SequenceClassifierOutput' object has no attribute 'backward'` | `out.backward()` 라고 썼다 → **`out.loss`** 로 손실을 꺼낸다 |
> | `CUDA out of memory` | 셀 3 의 `batch_size=32` 를 16 으로 |
> | 손실이 0.69 근처에서 안 내려간다 | 빈칸 ⑦ 순서 확인 (두 종류를 찍어 맞히는 수준의 손실이 약 0.69) |

---

## 5단계 — 평가: 혼동행렬과 틀린 문장

저장해 둔 **최고 시점 모델**을 폴더에서 다시 불러와 평가합니다. 정말로 저장이 됐는지 확인하는 과정이기도 합니다.

In [ ]:
# 셀 7 — 저장한 모델 다시 불러와 평가
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# 빈칸 ⑨ (1줄) — 힌트 : 저장 폴더 이름으로 불러온다.  AutoModelForSequenceClassification.from_pretrained(SAVE)
best = ____.to(device)
_, acc, preds = evaluate(best, val_loader)
print(f"저장한 모델의 검증 정확도 : {acc*100:.2f}%\n")
print(classification_report(va_y, preds, target_names=LABELS, digits=3))

ConfusionMatrixDisplay(confusion_matrix(va_y, preds), display_labels=LABELS).plot(cmap="Blues", colorbar=False)
plt.xlabel("모델의 예측"); plt.ylabel("정답"); plt.title("NSMC 검증 — KoELECTRA-small"); plt.show()

print("[틀린 문장 몇 개]")
wrong = [i for i, (y, p) in enumerate(zip(va_y, preds)) if y != p]
for i in wrong[:5]:
    print(f"  정답 {LABELS[va_y[i]]} → 예측 {LABELS[preds[i]]} | {va_x[i]}")

**기대 출력**
```
저장한 모델의 검증 정확도 : 80.10%

              precision    recall  f1-score   support

          부정      0.777     0.838     0.806       494
          긍정      0.829     0.765     0.795       506

    accuracy                          0.801      1000
   macro avg      0.803     0.801     0.801      1000
weighted avg      0.803     0.801     0.801      1000

[틀린 문장 몇 개]
  정답 긍정 → 예측 부정 | 아슈반 옛날생각나
  정답 긍정 → 예측 부정 | 아뭘자꾸공백을없애래10자썼잖아OO
  정답 긍정 → 예측 부정 | 지루하지만 그래도 볼만함
  정답 부정 → 예측 긍정 | 지금 조금 봤는데 쓰1레기 삘난다.
  정답 긍정 → 예측 부정 | 총기를 다른 걸로만 바꾸면 먼 나라 이야기가 아니다.
```

> **관찰**: 틀린 문장에는 ① 띄어쓰기·맞춤법이 무너진 문장(*아뭘자꾸공백을없애래10자썼잖아OO*), ② 칭찬과 불평이 섞인 문장(*지루하지만 그래도 볼만함*), ③ 감정 없이 내용만 말하는 문장(*총기를 다른 걸로만 바꾸면 먼 나라 이야기가 아니다.*)이 많습니다. 사람이 봐도 애매한 문장이 섞여 있습니다.

---

## 6단계 — 저장한 모델로 내 문장 판정

**`pipeline`** 은 *토크나이저 → 모델 → 점수를 확률로* 를 한 줄로 묶어 주는 도구입니다.
저장 폴더 이름만 주면 됩니다. 아래 `my_texts` 에 **내가 쓴 문장**을 넣어 보세요.

In [ ]:
# 셀 8 — pipeline 으로 내 문장 판정
# 빈칸 ⑩ (1줄) — 힌트 : 할 일 이름 "text-classification" 과 저장 폴더 SAVE 를 준다
clf = pipeline(____, model=____, device=0 if device.type == "cuda" else -1)

my_texts = [
    "이건 진짜 인생영화다",
    "돈 아까움 시간 낭비",
    "배우는 좋았는데 스토리가 아쉽네",
    "생각보다 괜찮았어요",
    "졸다가 나왔다",
]
for t, r in zip(my_texts, clf(my_texts)):
    print(f"{t:<20} → {r['label']} ({r['score']*100:.1f}%)")

**기대 출력**
```
이건 진짜 인생영화다          → 부정 (74.0%)
돈 아까움 시간 낭비          → 부정 (88.5%)
배우는 좋았는데 스토리가 아쉽네    → 부정 (79.4%)
생각보다 괜찮았어요           → 긍정 (92.8%)
졸다가 나왔다              → 부정 (82.2%)
```

> **관찰**: 훈련 3,000건·2 epoch(CPU 설정)에서는 **"이건 진짜 인생영화다"** 처럼 분명한 칭찬도 부정으로 틀렸고, 칭찬과 불평이 섞인 **"배우는 좋았는데 스토리가 아쉽네"** 는 부정으로 판정했습니다. 데이터와 학습이 적으면 이런 실수가 남습니다. 실습실 GPU 와 같은 조건(훈련 20,000건)으로 학습하면 같은 문장들이 이렇게 판정됩니다: *이건 진짜 인생영화다* → 긍정 (69.4%) · *돈 아까움 시간 낭비* → 부정 (98.6%) · *배우는 좋았는데 스토리가 아쉽네* → 부정 (92.5%) · *생각보다 괜찮았어요* → 긍정 (98.8%) · *졸다가 나왔다* → 부정 (95.7%).

> **해 볼 것**: 반어("참 잘~ 만들었네요")나 칭찬과 불평이 섞인 문장을 넣어 보세요. 어떤 문장에서 틀리나요?
> 이것이 과제의 "틀린 이유 추측" 재료입니다.

---

## 7단계 — 10주차 LSTM 과 비교

10주차 과제에서 얻은 **본인의 LSTM 검증 정확도**를 아래 `LSTM_ACC` 에 적고 실행합니다(예: `0.7655` — 10주차 정답본 실측 76.55%).
비워 두면(`None`) 이 노트북 결과만 표시합니다.

In [ ]:
# 셀 9 — 비교표  (빈칸 없음 — LSTM_ACC 만 바꿔 보세요)
LSTM_ACC = None                                   # 예) 0.7655  ← 10주차 내 결과 (정답본 실측 76.55%)

n_params = sum(p.numel() for p in best.parameters())
print(f"[KoELECTRA-small 파인튜닝]  파라미터 {n_params:,} | 훈련 {len(tr_x)}건 | 검증 정확도 {acc*100:.2f}%")
if LSTM_ACC is not None:
    print(f"[10주차 LSTM]               검증 정확도 {LSTM_ACC*100:.2f}%")
    print(f"차이 : {(acc - LSTM_ACC)*100:+.2f}%p")

**기대 출력** (`LSTM_ACC = None` 일 때)
```
[KoELECTRA-small 파인튜닝]  파라미터 14,122,498 | 훈련 3000건 | 검증 정확도 80.10%
```

> **핵심 ★**: 사전학습 모델은 **이미 수많은 한국어 문장으로 단어와 문맥을 배운 상태**에서 시작합니다.
> 그래서 **적은 데이터와 짧은 학습**으로도 처음부터 학습한 LSTM 보다 높은 정확도를 냅니다 — 9주차 전이학습과 같은 이야기입니다.

---

## 과제 — 파인튜닝 결과 + 미니프로젝트 주제 후보

| 제출물 | 내용 |
|---|---|
| ① 노트북 | 이 노트북을 **`16_text_finetune.ipynb`** 로 저장 (빈칸 완성 · 실행 결과 포함) |
| ② 비교 | 셀 9 결과 — 10주차 LSTM 과 정확도 비교 2줄 |
| ③ 내 문장 | 직접 쓴 문장 **5개**의 판정 결과와, 틀린 문장이 있다면 그 이유 추측 2줄 |
| ④ **프로젝트 주제 후보 2개** ★ | 아래 양식으로 2개 (12주차 첫 시간에 하나로 확정) |

```
[주제 후보 1]
- 분류 종류 : 이미지 / 텍스트            (택1)
- 무엇을 분류하나 : 예) 우리 집 고양이 3마리 구분, 상품 리뷰 긍정/부정/중립
- 클래스 : 예) 3개 — 나비 / 까미 / 치즈
- 데이터를 어디서 어떻게 모으나 : 예) 휴대폰 사진 클래스당 40장
- 왜 이 주제인가 (1줄) :
```

> 미니프로젝트는 **개인 수행**입니다. 이미지 분류(9주차 노트북 확장) 또는 텍스트 분류(이 노트북 확장) 중 하나를 권장합니다.
> 이미지는 클래스당 30장 이상, 텍스트는 클래스당 100문장 이상을 모을 수 있는 주제를 고르세요.

### 체크리스트

- [ ] 토크나이저가 문장을 서브워드로 자르는 것을 확인했다 (`[CLS]`, `[SEP]`, `##`)
- [ ] 사전학습 모델에 `num_labels=2` 분류 헤드를 붙였다
- [ ] HuggingFace 출력에서 `.logits` / `.loss` 를 꺼냈다
- [ ] 학습 루프를 직접 쓰고 최고 시점을 `save_pretrained` 로 저장했다
- [ ] 저장한 폴더에서 모델을 다시 불러와 평가했다
- [ ] `pipeline` 으로 내 문장을 판정했다
- [ ] 10주차 LSTM 과 비교했다